In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

ruta_actual = os.getcwd()
ruta_base = os.path.dirname(ruta_actual) if "notebooks" in ruta_actual else ruta_actual
ruta_csv = os.path.join(ruta_base, "data", "processed", "autos_limpios.csv")

df = pd.read_csv(ruta_csv)
df.head()

,titulo,precio,ano,kilometraje,marca
0,Chevrolet Tracker,4660056.0,2013.0,184171.0,Chevrolet
1,Kia Sorento,5827790.0,2016.0,171740.0,Kia
2,Chevrolet Silverado,10436160.0,2018.0,116856.0,Chevrolet
3,Chevrolet Tracker,16840408.0,2025.0,12105.0,Chevrolet
4,CHEVROLET SAIL,15258284.0,2022.0,57880.0,Chevrolet


In [ ]:
#Definir variables de entrada y salida
X = df[['ano', 'kilometraje', 'marca']]
y = df['precio']

print("Características de entrada (X):")
print(X.head())
print("\nVariable objetivo (y):")
print(y.head())

Características de entrada (X):
      ano  kilometraje      marca
0  2013.0     184171.0  Chevrolet
1  2016.0     171740.0        Kia
2  2018.0     116856.0  Chevrolet
3  2025.0      12105.0  Chevrolet
4  2022.0      57880.0  Chevrolet

Variable objetivo (y):
0     4660056.0
1     5827790.0
2    10436160.0
3    16840408.0
4    15258284.0
Name: precio, dtype: float64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas")

Datos de entrenamiento: 648 filas
Datos de prueba: 163 filas


In [ ]:
#Definir cómo tratar la columna categórica (marca)
transformador_categorico = OneHotEncoder(handle_unknown='ignore')

#Aplicar la transformación a columnas de texto
preprocesador = ColumnTransformer(
    transformers=[
        ('cat', transformador_categorico, ['marca'])
    ],
    remainder='passthrough'
)

#Crear el Pipeline completo usando el algoritmo RandomForest
pipeline_rf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', RandomForestRegressor(n_estimators=100, random_state=42))
])

#Entrenar el modelo
pipeline_rf.fit(X_train, y_train)
print("Modelo RandomForest entrenado con éxito")

Modelo RandomForest entrenado con éxito


In [ ]:
#Realizar las predicciones
predicciones = pipeline_rf.predict(X_test)

#Calcular métricas cuantitativas
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print("=== EVALUACIÓN DEL MODELO ===")
print(f"Error Absoluto Medio (MAE): ${mae:,.0f}".replace(",", "."))
print(f"Coeficiente de Determinación (R²): {r2:.2f}")
print("=============================")

=== EVALUACIÓN DEL MODELO ===
Error Absoluto Medio (MAE): $2.548.834
Coeficiente de Determinación (R²): 0.51


In [ ]:
#Ejemplo de Tasación
mi_auto = pd.DataFrame([{
    'ano': 2018,
    'kilometraje': 85000,
    'marca': 'Toyota'
}])

precio_estimado = pipeline_rf.predict(mi_auto)[0]

print(f"Tasación para un Toyota año 2018 con 85.000 Km:")
print(f"Precio estimado por el modelo: ${precio_estimado:,.0f}".replace(",", "."))

Tasación para un Toyota año 2018 con 85.000 Km:
Precio estimado por el modelo: $6.842.382
